In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
def charger_images(dossier):

    images = []
    labels = []

    femmes_path = os.path.join(dossier, "Femmes")
    hommes_path = os.path.join(dossier, "Hommes")
    for img_name in os.listdir(femmes_path):
        path = os.path.join(femmes_path, img_name)
        image = cv2.imread(path)

        if image is not None:
            images.append(image)
            labels.append(1)
    for img_name in os.listdir(hommes_path):
        path = os.path.join(hommes_path, img_name)
        image = cv2.imread(path)

        if image is not None:
            images.append(image)
            labels.append(0)

    return images, labels

Mounted at /content/drive


In [9]:
!unzip -n "/content/drive/MyDrive/deep/datasetImg.zip" -d "/content/"

images, labels = charger_images("/content/datasetImg")

Archive:  /content/drive/MyDrive/deep/datasetImg.zip


In [10]:
labels = np.array(labels)

nb_femmes = np.sum(labels == 1)
nb_hommes = np.sum(labels == 0)

print("Nombre d'images Femmes :", nb_femmes)
print("Nombre d'images Hommes :", nb_hommes)

Nombre d'images Femmes : 100
Nombre d'images Hommes : 98


In [11]:
def afficher_dimensions(images):

    tailles = set()

    for img in images:
        h, w, c = img.shape
        tailles.add((h, w, c))
        print("Hauteur:",h,"Largeur:",w,"Plans:",c)

    print("\nTailles différentes trouvées :", tailles)

In [12]:
afficher_dimensions(images)

Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 200 Largeur: 163 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur: 178 Plans: 3
Hauteur: 218 Largeur

In [13]:
def resize_images(images, size=(128,128)):

    resized = []

    for img in images:
        r = cv2.resize(img, size)
        resized.append(r)

    return resized

In [14]:
images_resized = resize_images(images)

In [15]:
X = []
y = labels

for img in images_resized:

    vecteur = img.reshape(-1)
    X.append(vecteur)

X = np.array(X)

print("Shape X :", X.shape)

Shape X : (198, 49152)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

pred = knn.predict(X_test)

acc1 = accuracy_score(y_test, pred)

print("Accuracy avant préparation :", acc1)

Accuracy avant préparation : 0.6333333333333333


In [17]:
def convertir_gris(images):

    images_gray = []

    for img in images:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        images_gray.append(gray)

    return images_gray
images_gray = convertir_gris(images_resized)

In [18]:
def filtre_median(images):

    resultat = []

    for img in images:
        filtered = cv2.medianBlur(img, 5)
        resultat.append(filtered)

    return resultat
images_filtre = filtre_median(images_gray)

In [19]:
def egalisation(images):

    result = []

    for img in images:
        eq = cv2.equalizeHist(img)
        result.append(eq)

    return result
images_final = egalisation(images_filtre)

In [20]:
X2 = []

for img in images_final:
    vecteur = img.reshape(-1)
    X2.append(vecteur)

X2 = np.array(X2)

In [21]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y,
    test_size=0.3,
    random_state=42
)
knn2 = KNeighborsClassifier(n_neighbors=5)

knn2.fit(X_train2, y_train2)

pred2 = knn2.predict(X_test2)

acc2 = accuracy_score(y_test2, pred2)

print("Accuracy après préparation :", acc2)

Accuracy après préparation : 0.6666666666666666


In [22]:
print("Performance avant préparation :", acc1)
print("Performance après préparation :", acc2)

Performance avant préparation : 0.6333333333333333
Performance après préparation : 0.6666666666666666
